In [0]:
%pip install langchain langchain-community databricks-langchain mlflow databricks-vectorsearch langchain-openai -q

dbutils.library.restartPython()

In [0]:
import mlflow
mlflow.set_experiment("/Users/jainanvesha14@gmail.com/saffron_shield")

mlflow.langchain.autolog()

In [0]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
import re

loader = DirectoryLoader(
    "/Volumes/claims/raw/landing/policy_docs/",
    glob = "**/*.md",
    loader_cls = TextLoader
)

docs = loader.load()

rows = []

for d in docs:
    filename = d.metadata["source"].split("/")[-1]
    full_text = d.page_content

    parts = re.split(r'(?m)^##\s+(.*)$', full_text)
    doc_header = parts[0].strip()
    sections = list(zip(parts[1::2], parts[2::2]))

    for idx, (heading, body) in enumerate(sections):
        heading_clean = heading.strip()
        body_clean = body.strip()
        anchor = re.sub(r'[^a-z0-9]+', '-', heading_clean.lower()).strip('-')
        chunk_id = f"{filename}#{anchor}"
        chunk_text = f"{doc_header}\n\n## {heading_clean}\n\n{body_clean}"

        rows.append({
            "chunk_id": chunk_id,
            "doc_name": filename,
            "heading": heading_clean,
            "chunk_index": idx,
            "text": chunk_text
        })


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("doc_name", StringType(), True),
    StructField("heading", StringType(), True),
    StructField("chunk_index", IntegerType(), True),
    StructField("text", StringType(), True),
])

new_chunks_df = spark.createDataFrame(rows, schema=schema)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS claims.genai.policy_chunks (
  chunk_id    STRING NOT NULL,
  doc_name    STRING,
  heading     STRING,
  chunk_index INT,
  text        STRING
)
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "claims.genai.policy_chunks")

target.alias("target").merge(
    new_chunks_df.alias("source"),
    "target.chunk_id = source.chunk_id"
).whenMatchedUpdate(
    condition = "target.text != source.text or target.heading != source.heading",
    set = {
        "text": "source.text",
        "heading": "source.heading"
    }
).whenNotMatchedInsert(
    values = {
        "chunk_id": "source.chunk_id",
        "doc_name": "source.doc_name",
        "heading": "source.heading",
        "chunk_index": "source.chunk_index",
        "text": "source.text"
    }
).whenNotMatchedBySourceDelete() \
.execute()

In [0]:
%sql
select * from claims.genai.policy_chunks

In [0]:
%sql 
describe history claims.genai.policy_chunks

In [0]:
from databricks.ai_search.client import AISearchClient
client = AISearchClient(disable_notice=True)

In [0]:
endpoint = "claims_vs"
client.create_endpoint(
    name = endpoint
)


In [0]:
client.get_endpoint(endpoint)

In [0]:
%sql
alter table claims.genai.policy_chunks set tblproperties (delta.enableChangeDataFeed = true)

In [0]:
index_name = "claims.genai.policy_chunk_index"
source_table = "claims.genai.policy_chunks"
embedding_model = "databricks-gte-large-en"

In [0]:


index = client.create_delta_sync_index(
    endpoint_name = endpoint,
    index_name = index_name,
    source_table_name = source_table,
    pipeline_type = "TRIGGERED",
    primary_key = "chunk_id",
    embedding_source_column = "text",
    embedding_model_endpoint_name = embedding_model
)

In [0]:
index.wait_until_ready()

# R3 Langchain basics

In [0]:
from databricks_langchain import ChatDatabricks
llm = ChatDatabricks(model = "databricks-meta-llama-3-3-70b-instruct")
response = llm.invoke("What is an health insurance policy")
print(response)

In [0]:
from langchain_openai import ChatOpenAI
llm2 = ChatOpenAI(
    model = "databricks-meta-llama-3-3-70b-instruct",
    base_url = "https://dbc-e63a3b85-204d.cloud.databricks.com/serving-endpoints",
    api_key = ""
)
response2 = llm2.invoke("What is an health insurance policy")
print(response2)
 

In [0]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(model = "databricks-meta-llama-3-3-70b-instruct")

class WaitingPeriodFacts(BaseModel):
    doc_name: str = Field(description="The file name, exactly as given, no folder path")
    plan_family: str = Field(description="individual or group, from the document's Applies to: line")
    initial_waiting_days: int = Field(description="Initial waiting period in days for this plan family")
    ped_waiting_months: int = Field(description="Pre-existing disease waiting period in months for this plan family")

parser = PydanticOutputParser(pydantic_object=WaitingPeriodFacts)
prompt = PromptTemplate(
    template = ("extract the waiting period fact from this pilicy document.\n"
    "{format_instructions}\n"
    "Documents: \n {document_text}"
    ),
    input_variables = ["document_text"],
    partial_variables = {"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser
waiting_facts = []
for filename in ["02_Waiting_Periods_Individual_Plans.md", "03_Waiting_Periods_Group_Plans.md"]:
    with open(f"/Volumes/claims/raw/landing/policy_docs/{filename}") as f:
        text = f.read()
    result = chain.invoke({"document_text": text})
    waiting_facts.append(result)
print(waiting_facts)

In [0]:
import json
output = {
    "chat_databricks_response" : response.content,
    "chat_openai_response" : response2.content,
    "waiting_facts" : [f.dict() for f in waiting_facts]
}
with open("/Workspace/Users/jainanvesha14@gmail.com/Saffron-Shield/evidance/r3_langchain.json", "w") as f:
    json.dump(output, f, indent=4)

# R4 Hybrid retriver + rerank

In [0]:
# Hybrid search 
from databricks_langchain import ChatDatabricks
def hybrid_search(query_text, k = 10):
    return index.similarity_search(
        query_text = query_text,
        columns = ["chunk_id", "doc_name", "heading", "text"],
        num_results = k,
        query_type = "HYBRID"   
    )

llm = ChatDatabricks(
    model = "databricks-meta-llama-3-3-70b-instruct",
    temperature = 0,
    max_tokens = 1024
)

In [0]:
# Rerank function 
import json
from langchain_core.messages import SystemMessage, HumanMessage

def rerank(query, hybrid_results):
    candidates = hybrid_results["result"]["data_array"]
    candidate_text = []
    for i, row in enumerate(candidates):
        candidate_text.append({
            "candidate_id": i,
            "doc_name": row[1],
            "heading": row[2],
            "text": row[3]
        })

    rerank_prompt = f"""
You are a search relevance reranker for insurance policy documents.
User query:
{query}
Candidate policy sections:
{json.dumps(candidate_text, indent=2)}
Rank the candidates from most relevant to least relevant.
Rules:

* Consider the exact user question.
* Prefer the correct plan type (individual or group).
* Prefer sections directly answering the question.
* Do not invent information.
* Return ONLY valid JSON.
* Return this format:

{{"ranked_ids": [0, 1, 2, 3, 4]}}

* Include every candidate ID exactly once.

"""
    response = llm.invoke([
        SystemMessage(content="You are precise search reranking engine."),
        HumanMessage(content=rerank_prompt)
    ])

    raw = response.content.strip()
    json_start = raw.find("{")
    json_end = raw.rfind("}")
    ranked_ids = json.loads(raw[json_start:json_end + 1])["ranked_ids"]
    if sorted(ranked_ids) != list(range(len(candidates))):
        raise ValueError("Invalid ranking returned by the LLM")

    reranked_results = [candidates[i] for i in ranked_ids]
    return reranked_results
        

In [0]:
def hybrid_search_and_rerank(query_text, top_k=5):
    hybrid_results = hybrid_search(query_text, k = 10)
    reranked = rerank(query_text, hybrid_results)
    return reranked[:top_k]

In [0]:
index = client.get_index(
    endpoint_name="claims_vs",
    index_name="claims.genai.policy_chunk_index"
)

In [0]:
test_queries = {
    "G04": "How long is the initial waiting period on a Silver or Gold individual plan?",
    "G05": "What is the pre-existing disease waiting period for an individual plan?",
    "G07": "Is there an initial waiting period on a corporate group plan bought by an employer?",
    "G08": "Is there an initial waiting period on a corporate group plan bought by an employer",
}

index = client.get_index(
    endpoint_name="claims_vs",
    index_name="claims.genai.policy_chunk_index"
)

for qid, qtext in test_queries.items():
    top5 = hybrid_search_and_rerank(qtext)
    top_doc = top5[0][1] 
    print(qid, "->", top_doc)

# R5 RAG chain

In [0]:
from langchain_core.prompts import PromptTemplate
def policy_response(query):
    top_chunks = hybrid_search_and_rerank(query)

    context_blocks = []
    for row in top_chunks:
        chunk_id, doc_name, heading, text = row[:4]
        context_blocks.append(f"{chunk_id} \n {text}")

    context = "\n\n".join(context_blocks)

    prompt = PromptTemplate(
        input_variables = ["context", "query"],
        template = """
        You are answering a question about an insurance policy binder, using ONLY the
        retrieved sections below. Do not use any outside knowledge.

        If the retrieved sections do not contain the answer, respond with EXACTLY
        the phrase: not in policy
        Do not use that phrase for any other reason.

        Question: {query}

        Retrieved sections:
        {context}

        Answer:
        """
    )
    response = llm.invoke([
        SystemMessage(content = "Your are policy knowledge assistance that stricly answer from the provided context policy text and if the answe is not in the text then respond with :not in policy"),
        HumanMessage(content = prompt.format(context = context, query = query))
    ])

    answer_text = str(response.content).strip()
    cited_ids = [row[0] for row in top_chunks]

    return {
        "answer" : answer_text,
        "cited_chunk_ids" : cited_ids
    }

In [0]:
import json

with open("/Volumes/claims/raw/landing/gold_set.json") as f:
    gold_set = json.load(f)   # your existing gold-set questions

rows = []

# process all gold-set questions
for item in gold_set["questions"]:
    result = policy_response(item["question"])
    rows.append({
        "question_id": item["id"],
        "question": item["question"],
        "answer": result["answer"],
        "cited_chunk_ids": result["cited_chunk_ids"]
    })

# add the two explicitly unanswerable questions
unanswerable = [
    {"question_id": "U01", "question": "What is the premium for adding a pet dog to a Saffron Shield plan?"},
    {"question_id": "U02", "question": "Does Saffron Shield pay for a member's gym membership?"},
]

for item in unanswerable:
    result = policy_response(item["question"])
    rows.append({
        "question_id": item["question_id"],
        "question": item["question"],
        "answer": result["answer"],
        "cited_chunk_ids": result["cited_chunk_ids"]
    })

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

schema = StructType([
    StructField("question_id", StringType(), False),
    StructField("question", StringType(), True),
    StructField("answer", StringType(), True),
    StructField("cited_chunk_ids", ArrayType(StringType()), True),
])

df = spark.createDataFrame(rows, schema=schema)
df.write.mode("overwrite").saveAsTable("claims.genai.gold_answers")  

# R6 Converstion memory 

In [0]:
def update_summary(previous_summary, question, answer):
    prompt = f"""
    Update this running conversation summary to include the latest question and answer pair. 
    keep it brief and concise but covering the key context in the running conversation 
    Previous summary: {previous_summary or "none yet"}
    New question: {question}
    New Answer: {answer}
    """

    response = llm.invoke([
        SystemMessage(content = "your main task is to maintain an short factual summary of the running conversation"),
        HumanMessage(content = prompt)
    ])

    return (response.content).strip()

In [0]:
def make_standalone(question, summary):
    if not summary:
        return question

    prompt = f"""
    Conversation summary: {summary}

    Follow up Question: {question}
    Rewrite the follow up question to be a fully standalone question taht makes sense without the conversation history. Do not answer it only rewrite the question.
    return only the standalone question without any supporting text.
    """

    response = llm.invoke([
        SystemMessage(content = "Rewite the follow up question into standalone question using conversation summary as context"),
        HumanMessage(content = prompt)
    ])

    return (response.content).strip()

In [0]:
def run_chat(turn_number, question, state):
    standalone_q = make_standalone(question, state["summary"])

    result = policy_response(standalone_q)

    state["summary"] = update_summary(state["summary"], standalone_q, result["answer"])

    turn_record = {
        "turn" : turn_number,
        "question": question,
        "standalone_question": standalone_q,
        "answer": result["answer"],
        "cited_chunk_ids": result["cited_chunk_ids"]
    }

    return turn_record, state

In [0]:
state = {"summary": None}
turns = []
script = [
    "What is the pre-existing disease waiting period for individual plans?",
    "What about the initial waiting period?",
    "And for group plans?"
]

for i, q in enumerate(script, start = 1):
    trun_record, state = run_chat(i,q,state)
    turns.append(trun_record)

In [0]:
import mlflow

mlflow.langchain.autolog()

with mlflow.start_run() as run:
    with mlflow.start_span(name="three_turn_chat") as span:
        state = {"summary": None}
        turns = []
        for i, q in enumerate(script, start=1):
            turn_record, state = run_chat(i, q, state)
            turns.append(turn_record)
            print(turn_record)
        trace_id = mlflow.get_last_active_trace_id()

In [0]:
import json

output = {
    "turns": turns,          
    "summary": state["summary"],   
    "trace_id": trace_id 
}

assert len(output["summary"]) >= 40, "summary too short"
assert output["trace_id"], "trace_id is empty"
assert len(output["turns"]) == 3

with open("/Workspace/Users/jainanvesha14@gmail.com/Saffron-Shield/evidance/r6_memory.json", "w") as f:
    json.dump(output, f, indent=2)

# R7 Evaluation and Observability

In [0]:
def hit_rate_at_5(gold_questions):
    hits = 0
    for item in gold_questions:
        result = policy_response(item["question"])
        if any(chunk in result["cited_chunk_ids"] for chunk in item["expected_chunks"]):
            hits += 1
    return hits / len(gold_questions)


def mrr(gold_questions):
    reciprocal_ranks = []
    for item in gold_questions:
        result = policy_response(item["question"])
        cited = result["cited_chunk_ids"]
        ranks = [cited.index(chunk) + 1 for chunk in item["expected_chunks"] if chunk in cited]
        if ranks:
            reciprocal_ranks.append(1 / min(ranks))
        else:
            reciprocal_ranks.append(0)
    return sum(reciprocal_ranks) / len(reciprocal_ranks)

In [0]:
import json

with open("/Volumes/claims/raw/landing/gold_set.json") as f:
    gold_set = json.load(f) 
    
single_doc_questions = [
    q for q in gold_set["questions"]
    if not str(q["expected_chunks"]).startswith("CROSS_SOURCE__")
]

hr5 = hit_rate_at_5(single_doc_questions)
mrr_score = mrr(single_doc_questions)
n_questions = len(single_doc_questions)

print("hit-rate@5:", hr5, "(threshold 0.80)")
print("MRR:", mrr_score, "(threshold 0.70)")
print("Number of Questions:", n_questions)

In [0]:
import mlflow
from mlflow.genai.scorers import RetrievalGroundedness, RelevanceToQuery, Correctness

try:
    eval_dataset = [
        {
            "inputs": {"question": item["question"]},
            "expectations": {"expected_facts": item.get("expected_facts", [])}
        }
        for item in gold_set["questions"]  # or single_doc_questions, depending on scope you want judged
    ]

    # The judges need one trace per question with a RETRIEVER span; the vector search
    # is not instrumented, so wrap the retriever just for this evaluation.
    _untraced_hybrid_search_and_rerank = hybrid_search_and_rerank

    @mlflow.trace(span_type="RETRIEVER", name="policy_retriever")
    def _traced_hybrid_search_and_rerank(query_text, top_k=5):
        chunks = _untraced_hybrid_search_and_rerank(query_text, top_k)
        span = mlflow.get_current_active_span()
        if span is not None:
            span.set_outputs(
                [{"doc_name": row[1], "heading": row[2], "text": row[3]} for row in chunks]
            )
        return chunks

    @mlflow.trace(name="policy_response")
    def predict_fn(question):
        result = policy_response(question)
        return result["answer"]

    hybrid_search_and_rerank = _traced_hybrid_search_and_rerank
    try:
        with mlflow.start_run() as run:
            eval_results = mlflow.genai.evaluate(
                data=eval_dataset,
                predict_fn=predict_fn,
                scorers=[RetrievalGroundedness(), RelevanceToQuery(), Correctness()]
            )
            eval_run_id = run.info.run_id
            experiment_id = run.info.experiment_id
            judges_status = "ran"
            judges = eval_results.metrics
    finally:
        hybrid_search_and_rerank = _untraced_hybrid_search_and_rerank

except Exception as e:
    eval_run_id = None
    active_run = mlflow.active_run()
    experiment_id = (
        active_run.info.experiment_id
        if active_run
        else mlflow.get_experiment_by_name(
            "/Users/jainanvesha14@gmail.com/Saffron-Shield/genai/Retrival and langchain"
        ).experiment_id
    )
    judges_status = f"failed: {str(e)}"
    judges = {}

In [0]:
import json

eval_output = {
    "hit_rate_at_5": hr5,
    "mrr": mrr_score,
    "n_questions": n_questions,
    "mlflow_experiment_id": experiment_id,
    "eval_run_id": eval_run_id,
    "judges_status": judges_status,
    "judges": judges
}

with open("/Workspace/Users/jainanvesha14@gmail.com/Saffron-Shield/evidance/r7_eval.json", "w") as f:
    json.dump(eval_output, f, indent=2)

print(json.dumps(eval_output, indent=2))

# R8 ai_query and Adjudication

In [0]:
%sql
CREATE OR REPLACE TABLE claims.genai.adjudication_probes AS
SELECT claim_id
FROM read_files(
  '/Volumes/claims/raw/landing/adjudication_probe_claims.csv',
  format => 'csv',
  header => true
);

In [0]:
%sql
CREATE OR REPLACE TABLE claims.genai.adjudication_verifier AS

WITH probe_claims AS (
  SELECT f.claim_id, f.member_sk, f.room_rent_over_cap, f.is_network_at_admission,
         sc.admission_date, pl.plan_sk, pl.policy_id, pl.member_id, pl.plan_code
  FROM claims.genai.adjudication_probes p
  JOIN claims.analytics.fact_claims f ON p.claim_id = f.claim_id
  JOIN claims.silver.claims_clean sc ON p.claim_id = sc.claim_id
  JOIN claims.analytics.dim_plan pl ON f.plan_sk = pl.plan_sk
),

enriched AS (
  SELECT
    pc.claim_id,
    pc.admission_date,
    pc.is_network_at_admission,

    -- member's first-ever policy start date (across all their plan versions)
    (SELECT MIN(effective_from) FROM claims.analytics.dim_plan WHERE member_id = pc.member_id) AS first_policy_start,

    rp.initial_waiting_days,
    rp.ped_waiting_months,
    rp.copay_pct AS base_copay_pct,
    rp.room_rent_cap_per_day,

    f.room_rent_per_day,  -- claim-level detail sourced from claims.silver.claims_clean
    f.diagnosis_code,
    f.is_pre_existing

  FROM probe_claims pc
  JOIN claims.raw.plans rp ON pc.plan_code = rp.plan_code
  JOIN claims.silver.claims_clean f ON pc.claim_id = f.claim_id
),

calculated AS (
  SELECT
    claim_id,

    -- initial waiting: days since first policy start >= initial_waiting_days
    (DATEDIFF(admission_date, first_policy_start) >= initial_waiting_days) AS initial_waiting_ok,

    -- PED waiting: not a pre-existing diagnosis, OR tenure in completed months >= ped_waiting_months
    (
      NOT is_pre_existing
      OR
      (MONTHS_BETWEEN(admission_date, first_policy_start) >= ped_waiting_months)
    ) AS ped_waiting_ok,

    is_network_at_admission,

    -- room rent deduction: 0 if within cap, else (1 - cap/room_rent)*100
    CASE
      WHEN room_rent_per_day <= room_rent_cap_per_day THEN 0.0
      ELSE ROUND((1 - (room_rent_cap_per_day * 1.0 / room_rent_per_day)) * 100, 2)
    END AS room_rent_deduction_pct,

    -- copay: base + 20 if out of network
    base_copay_pct + CASE WHEN is_network_at_admission THEN 0 ELSE 20 END AS copay_pct

  FROM enriched
)

SELECT
  claim_id,
  (initial_waiting_ok AND ped_waiting_ok) AS waiting_period_ok,
  (initial_waiting_ok AND ped_waiting_ok) AS payable,   -- payable = waiting_period_ok, per Definitions (no exclusions in this data)
  is_network_at_admission AS network_at_admission,
  room_rent_deduction_pct,
  copay_pct,
  CASE
    WHEN NOT initial_waiting_ok THEN 'INITIAL_WAITING'
    WHEN NOT ped_waiting_ok THEN 'PED_WAITING'
    WHEN (initial_waiting_ok AND ped_waiting_ok) AND (room_rent_deduction_pct > 0 OR copay_pct > 0) THEN 'PAYABLE_WITH_DEDUCTION'
    ELSE 'PAYABLE_IN_FULL'
  END AS reason_code
FROM calculated;

In [0]:
%sql
CREATE OR REPLACE TABLE claims.genai.adjudications AS

WITH claim_data AS (
  SELECT
    pc.claim_id,
    pc.admission_date,
    pc.is_network_at_admission,
    (SELECT MIN(effective_from) FROM claims.analytics.dim_plan WHERE member_id = pc.member_id) AS first_policy_start,
    rp.initial_waiting_days,
    rp.ped_waiting_months,
    rp.copay_pct AS base_copay_pct,
    rp.room_rent_cap_per_day,
    f.room_rent_per_day,
    f.is_pre_existing
  FROM (SELECT p.claim_id, f.is_network_at_admission, sc.admission_date, pl.member_id, pl.plan_code
        FROM claims.genai.adjudication_probes p
        JOIN claims.analytics.fact_claims f ON p.claim_id = f.claim_id
        JOIN claims.silver.claims_clean sc ON p.claim_id = sc.claim_id
        JOIN claims.analytics.dim_plan pl ON f.plan_sk = pl.plan_sk) pc
  JOIN claims.raw.plans rp ON pc.plan_code = rp.plan_code
  JOIN claims.silver.claims_clean f ON pc.claim_id = f.claim_id
)

SELECT
  claim_id,
  ai_query(
    'system.ai.meta-llama-3-3-70b-instruct',
    CONCAT(
      'You are adjudicating an insurance claim. Use ONLY these rules:\n',
      '1. Initial waiting passes when days from first policy start to admission date >= initial_waiting_days.\n',
      '2. PED waiting passes when the diagnosis is not pre-existing, OR completed months of tenure >= ped_waiting_months.\n',
      '3. waiting_period_ok = initial waiting passes AND PED waiting passes.\n',
      '4. payable = waiting_period_ok (no other exclusions apply).\n',
      '5. room_rent_deduction_pct = 0 if room_rent_per_day <= cap, else round((1 - cap/room_rent_per_day)*100, 2).\n',
      '6. copay_pct = base copay + 20 if not in-network at admission.\n',
      '7. reason_code: INITIAL_WAITING if rule 1 fails; else PED_WAITING if rule 2 fails; ',
         'else PAYABLE_WITH_DEDUCTION if payable and (deduction>0 or copay>0); else PAYABLE_IN_FULL.\n\n',
      'Worked example 1 (made-up claim): first_policy_start=2024-01-01, admission_date=2024-01-10, ',
      'initial_waiting_days=30, is_pre_existing=false, ped_waiting_months=24, room_rent_per_day=2000, ',
      'cap=2000, base_copay_pct=0, is_network_at_admission=true. ',
      'Result: initial waiting FAILS (9 days < 30). reason_code=INITIAL_WAITING, payable=false, ',
      'waiting_period_ok=false, deduction=0, copay_pct=0.\n\n',
      'Worked example 2 (made-up claim): first_policy_start=2022-01-01, admission_date=2024-06-01, ',
      'initial_waiting_days=30, is_pre_existing=true, ped_waiting_months=24, room_rent_per_day=3000, ',
      'cap=2500, base_copay_pct=10, is_network_at_admission=false. ',
      'Result: initial waiting passes. PED waiting passes (29 months >= 24). waiting_period_ok=true, ',
      'payable=true. deduction=round((1-2500/3000)*100,2)=16.67. copay_pct=10+20=30. ',
      'reason_code=PAYABLE_WITH_DEDUCTION.\n\n',
      'Now adjudicate this actual claim:\n',
      'first_policy_start=', first_policy_start, ', admission_date=', admission_date, ', ',
      'initial_waiting_days=', initial_waiting_days, ', ped_waiting_months=', ped_waiting_months, ', ',
      'is_pre_existing=', is_pre_existing, ', room_rent_per_day=', room_rent_per_day, ', ',
      'room_rent_cap_per_day=', room_rent_cap_per_day, ', base_copay_pct=', base_copay_pct, ', ',
      'is_network_at_admission=', is_network_at_admission, '.\n\n',
      'Return ONLY a JSON object with these exact keys: payable (boolean), waiting_period_ok (boolean), ',
      'network_at_admission (boolean), room_rent_deduction_pct (number), copay_pct (integer), ',
      'reason_code (string: one of INITIAL_WAITING, PED_WAITING, PAYABLE_WITH_DEDUCTION, PAYABLE_IN_FULL), ',
      'reasoning (string: your step-by-step reasoning).'
    ),
    responseFormat => '{
      "type": "json_schema",
      "json_schema": {
        "name": "adjudication",
        "schema": {
          "type": "object",
          "properties": {
            "payable": {"type": "boolean"},
            "waiting_period_ok": {"type": "boolean"},
            "network_at_admission": {"type": "boolean"},
            "room_rent_deduction_pct": {"type": "number"},
            "copay_pct": {"type": "integer"},
            "reason_code": {"type": "string"},
            "reasoning": {"type": "string"}
          },
          "required": ["payable","waiting_period_ok","network_at_admission","room_rent_deduction_pct","copay_pct","reason_code","reasoning"]
        }
      }
    }',
    modelParameters => named_struct('temperature', 0.0)
  ) AS response_json
FROM claim_data;

In [0]:
%sql
-- The raw ai_query JSON currently lives in claims.genai.adjudications (written by the cell above).
-- Preserve it in the staging table this cell parses from, then rebuild the final table with typed columns.
CREATE OR REPLACE TABLE claims.genai._staging_adjudications AS
SELECT claim_id, response_json
FROM claims.genai.adjudications;

CREATE OR REPLACE TABLE claims.genai.adjudications AS
SELECT
  claim_id,
  get_json_object(response_json, '$.payable')::boolean AS payable,
  get_json_object(response_json, '$.waiting_period_ok')::boolean AS waiting_period_ok,
  get_json_object(response_json, '$.network_at_admission')::boolean AS network_at_admission,
  get_json_object(response_json, '$.room_rent_deduction_pct')::double AS room_rent_deduction_pct,
  get_json_object(response_json, '$.copay_pct')::int AS copay_pct,
  get_json_object(response_json, '$.reason_code') AS reason_code,
  get_json_object(response_json, '$.reasoning') AS reasoning
FROM claims.genai._staging_adjudications; 

In [0]:
%sql
WITH compared AS (
  SELECT
    v.claim_id,
    (v.payable = a.payable
     AND v.waiting_period_ok = a.waiting_period_ok
     AND v.network_at_admission = a.network_at_admission
     AND v.copay_pct = a.copay_pct
     AND v.reason_code = a.reason_code
     AND ABS(v.room_rent_deduction_pct - a.room_rent_deduction_pct) <= 1.0
    ) AS agrees
  FROM claims.genai.adjudication_verifier v
  JOIN claims.genai.adjudications a ON v.claim_id = a.claim_id
)
SELECT
  ROUND(AVG(CASE WHEN agrees THEN 1.0 ELSE 0 END), 4) AS overall_agreement_rate,
  COUNT(*) AS total_claims,
  SUM(CASE WHEN agrees THEN 1 ELSE 0 END) AS agreeing_claims
FROM compared;